# VascuLens AI — CAD Model Benchmark

Benchmark clinical tabular classifiers for overall CAD using the Extension of Z-Alizadeh Sani dataset. The input matrix excludes `Cath`, `CAD`, `LAD`, `LCX`, and `RCA` to avoid leakage. The metrics are internal estimates; do not claim a target performance before running this notebook on the actual source data.

Research/education only; not a medical device. Dataset is not included.

In [ ]:
import os,glob,re,json,warnings
from pathlib import Path
import numpy as np,pandas as pd, matplotlib.pyplot as plt, joblib
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier,RandomForestClassifier,RandomForestClassifier
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from sklearn.metrics import accuracy_score,balanced_accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,brier_score_loss,confusion_matrix,roc_curve,precision_recall_curve
from sklearn.base import clone
warnings.filterwarnings('ignore'); SEED=42; np.random.seed(SEED)


In [ ]:
files=[]
for root in ['/kaggle/input','/kaggle/working','.']:
    if os.path.exists(root):
        for pat in ('**/*.xlsx','**/*.xls','**/*.csv','**/*.parquet'): files.extend(glob.glob(os.path.join(root,pat),recursive=True))
files=sorted(set(files))
print('Candidate files:',files[:40])
wanted=[p for p in files if any(k in os.path.basename(p).lower() for k in ('z-alizadeh','z_alizadeh','extention','alizadeh sani'))]
if not wanted: raise FileNotFoundError('Attach the original Extension of Z-Alizadeh Sani CSV/XLSX to Kaggle input.')
DATA_PATH=wanted[0]
raw=pd.read_csv(DATA_PATH) if DATA_PATH.lower().endswith(('.csv','.parquet')) else pd.read_excel(DATA_PATH,sheet_name=0)
raw.columns=[str(c).replace('\\ufeff','').strip() for c in raw.columns]
print(raw.shape,DATA_PATH); display(raw.head())


In [ ]:
def norm(x): return re.sub(r'[^a-z0-9]+','',str(x).lower())
lookup={norm(c):c for c in raw.columns}
target_col=lookup.get('cath',lookup.get('cad'))
if target_col is None: raise ValueError('Cannot find Cath or CAD target; inspect column names.')
positive={'cad','yes','positive','present','disease','stenosis','stenotic','true','1','1.0'}
negative={'normal','no','negative','absent','healthy','false','0','0.0'}
z=raw[target_col].astype(str).str.strip().str.lower(); unseen=set(z.unique())-positive-negative
if unseen: raise ValueError(f'Unmapped target labels {sorted(unseen)}; explicitly map them before proceeding.')
y=z.isin(positive).astype(int)
excluded={'cath','cad','lad','lcx','rca','target','label','class','diagnosis','outcome','result','y','patientid','id'}
feature_cols=[c for c in raw.columns if norm(c) not in excluded and not norm(c).startswith('unnamed')]
X=raw[feature_cols].copy()
for c in X.columns:
    if X[c].dtype=='object':
        n=pd.to_numeric(X[c],errors='coerce'); valid=X[c].notna().sum()
        if valid and n.notna().sum()/valid>=.95: X[c]=n
assert not ({norm(c) for c in X.columns}&excluded)
print('Target:',target_col,'class counts:',y.value_counts().to_dict(),'features:',len(feature_cols),'excluded:',sorted(excluded))
display(pd.DataFrame({'feature':X.columns,'dtype':[str(X[c].dtype) for c in X.columns],'missing':[int(X[c].isna().sum()) for c in X.columns]}))


In [ ]:
def prep(X):
    num=X.select_dtypes(include=['number','bool']).columns.tolist(); cat=[c for c in X.columns if c not in num]; tr=[]
    if num: tr.append(('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num))
    if cat: tr.append(('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat))
    return ColumnTransformer(tr)
models={
 'Logistic Regression':Pipeline([('prep',prep(X)),('model',LogisticRegression(max_iter=3000,class_weight='balanced',C=.5,random_state=SEED))]),
 'RBF SVM':Pipeline([('prep',prep(X)),('model',SVC(C=1,kernel='rbf',probability=True,class_weight='balanced',random_state=SEED))]),
 'Extra Trees':Pipeline([('prep',prep(X)),('model',ExtraTreesClassifier(n_estimators=500,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=1))]),
 'Random Forest':Pipeline([('prep',prep(X)),('model',RandomForestClassifier(n_estimators=500,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=1))])}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED); rows=[]; oof={}
for name,model in models.items():
    prob=cross_val_predict(clone(model),X,y,cv=cv,method='predict_proba',n_jobs=1)[:,1]; pred=(prob>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    oof[name]=prob; rows.append({'model':name,'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),'precision':precision_score(y,pred,zero_division=0),'sensitivity_recall':recall_score(y,pred,zero_division=0),'specificity':tn/(tn+fp) if tn+fp else np.nan,'F1':f1_score(y,pred,zero_division=0),'ROC_AUC':roc_auc_score(y,prob),'PR_AUC_average_precision':average_precision_score(y,prob),'Brier_score_lower_is_better':brier_score_loss(y,prob),'TN':tn,'FP':fp,'FN':fn,'TP':tp})
results_df=pd.DataFrame(rows).sort_values('ROC_AUC',ascending=False); display(results_df.round(3))


In [ ]:
plt.figure(figsize=(7,6))
for name,prob in oof.items():
    fpr,tpr,_=roc_curve(y,prob); plt.plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(y,prob):.3f})')
plt.plot([0,1],[0,1],'--',label='Chance'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.title('Out-of-fold ROC curves'); plt.legend(); plt.grid(alpha=.2); plt.show()
best_name=results_df.iloc[0]['model']; best_model=clone(models[best_name]).fit(X,y)
out=Path('/kaggle/working/vasculens_cad_benchmark'); out.mkdir(exist_ok=True,parents=True)
joblib.dump({'model':best_model,'features':feature_cols,'target_column':target_col},out/'cad_model.joblib')
results_df.to_csv(out/'benchmark_metrics.csv',index=False)
meta={'project':'VascuLens AI','target':target_col,'model_selected_by':'OOF ROC-AUC','selected_model':best_name,'feature_count':len(feature_cols),'excluded_target_columns':sorted(excluded),'seed':SEED,'validation':'stratified 5-fold out-of-fold predictions','clinical_warning':'Research only; small single-source dataset; not external validation.'}
(out/'metadata.json').write_text(json.dumps(meta,indent=2)); print('Saved artifacts:',out); print(meta)


## Interpretation

Metrics in this notebook are out-of-fold cross-validation estimates on a small single-source dataset. The best-ranked model by ROC-AUC is not necessarily the best by accuracy or sensitivity. Do not claim 95% accuracy unless it is observed and reported with a suitable validation design. Train/test separation, feature leakage control, threshold selection and external validation matter more than tuning to one headline number. The fitted model in the output folder is trained on all rows after benchmarking and is not a test-performance estimate.